# Multilingual OCR — PP-OCRv6 + HI-RES reading order (self-contained)

Brings the HI-RES deterministic **reading-order reconstruction** stage to multilingual document
OCR, using **PP-OCRv6** (June 2026 — one unified model for 50 languages) for *both* detection and
recognition, on both sides:

```
PP-OCRv6 detection → reading-order reconstruction → PP-OCRv6 recognition
        vs   stock PaddleOCR running the same PP-OCRv6 models
```

Two experiments:
1. **Clean forms (XFUND)** — controlled accuracy + speed at equal models (expect ~parity; the
   layer's batched recognition is the speed edge).
2. **Skew robustness** — rotate the pages and watch stock's reading order break while HI-RES
   holds. *This* is the layer's real value and the basis for any PaddleOCR contribution.

**Runtime → Change runtime type → T4 GPU** strongly recommended. Self-contained: embeds the
source, auto-downloads XFUND, runs top to bottom.

## 1. Install dependencies (~2 min)

In [ ]:
%pip install -q -U paddleocr paddlepaddle datasets opencv-python-headless
import paddleocr
print('paddleocr', paddleocr.__version__, '(need a build with PP-OCRv6, i.e. mid-2026+)')

## 2. Project source (embedded)
Geometry (`pipeline.py`), the paddle-only **detector** (`detector.py`), metrics (`evaluate.py`),
then the two multilingual modules. **No TrOCR / handwriting engine here** — detection and
recognition are both PP-OCR, so this notebook never imports `ocr_engine.py`.

In [ ]:
%%writefile pipeline.py
"""Pure geometry for the OCR pipeline: reading order, line clustering, cropping.

Everything in this module is deterministic and depends only on numpy/cv2,
so it can be unit-tested without loading any model.

Coordinate convention: image coordinates, x right, y DOWN. Quads are 4x2
arrays ordered [top-left, top-right, bottom-right, bottom-left] after
passing through `order_points`.

Reading-order algorithm (per page):
  1. Estimate global skew as the median angle of the detected boxes'
     horizontal edges (wide boxes only, when enough exist).
  2. Rotate all box corners by the negative skew -> "deskewed space" where
     text lines are horizontal. Only coordinates are rotated, never pixels.
  3. Conservatively split into column blocks: a vertical cut requires a
     projection gap wider than `gap_factor * median box height` with no box
     crossing it, enough boxes on both sides, and both sides spanning most
     of the block height. Recurses (depth-limited) for 3+ columns.
  4. Within each block, cluster boxes into lines: a box joins the line whose
     vertical band it overlaps by >= `overlap_thresh` of the smaller height.
  5. Order lines by band center y; order boxes within a line by center x.
  6. Optionally split very long lines into chunks whose aspect ratio TrOCR
     can handle (it squeezes every crop to a 384x384 square).
"""

from __future__ import annotations

from dataclasses import dataclass, field

import cv2
import numpy as np

__all__ = [
    "order_points",
    "quad_size",
    "estimate_skew",
    "rotate_points",
    "split_columns",
    "cluster_lines",
    "reading_order",
    "chunk_line",
    "merge_quads",
    "expand_quad",
    "perspective_crop",
    "assemble_text",
    "annotate",
    "compose_transcript",
    "Line",
]


# --------------------------------------------------------------------------
# basic quad utilities
# --------------------------------------------------------------------------

def order_points(quad) -> np.ndarray:
    """Return the 4 points ordered [TL, TR, BR, BL].

    Valid for quads rotated less than ~45 degrees, which is all this
    pipeline handles (page-level 90/180 rotation is an upstream
    orientation problem, not a reading-order problem).
    """
    pts = np.asarray(quad, dtype=np.float64).reshape(4, 2)
    idx = np.lexsort((pts[:, 1], pts[:, 0]))  # by x, ties by y
    left, right = pts[idx[:2]], pts[idx[2:]]
    tl, bl = left[np.argsort(left[:, 1])]
    tr, br = right[np.argsort(right[:, 1])]
    return np.array([tl, tr, br, bl])


def quad_size(quad) -> tuple[float, float]:
    """(width, height) as the mean of opposite edge lengths of an ordered quad."""
    q = np.asarray(quad, dtype=np.float64)
    w = (np.linalg.norm(q[1] - q[0]) + np.linalg.norm(q[2] - q[3])) / 2.0
    h = (np.linalg.norm(q[3] - q[0]) + np.linalg.norm(q[2] - q[1])) / 2.0
    return float(w), float(h)


def rotate_points(pts: np.ndarray, theta_deg: float) -> np.ndarray:
    """Rotate points by theta_deg around the origin (standard math rotation;
    with y-down image coords a positive theta maps a +theta-skewed page back
    to horizontal when applied as -theta)."""
    t = np.deg2rad(theta_deg)
    c, s = np.cos(t), np.sin(t)
    rot = np.array([[c, -s], [s, c]])
    return np.asarray(pts, dtype=np.float64) @ rot.T


def estimate_skew(quads: list[np.ndarray], max_abs_deg: float = 30.0,
                  wide_factor: float = 1.5) -> float:
    """Median angle (degrees) of box top/bottom edges.

    Prefers wide boxes (width >= wide_factor * height) because their edge
    direction reliably follows the text baseline; a near-square box around a
    single short word carries almost no angle information. Returns 0.0 when
    the estimate exceeds max_abs_deg — that signals a rotated page, which
    coordinate deskew must not attempt to fix.
    """
    angles, is_wide = [], []
    for quad in quads:
        q = order_points(quad)
        w, h = quad_size(q)
        if w < 2 or h < 2:
            continue
        v = ((q[1] - q[0]) + (q[2] - q[3])) / 2.0  # mean of top and bottom edge
        angles.append(np.degrees(np.arctan2(v[1], v[0])))
        is_wide.append(w >= wide_factor * h)
    if not angles:
        return 0.0
    angles = np.array(angles)
    wide = angles[np.array(is_wide)]
    med = float(np.median(wide if wide.size >= 3 else angles))
    return med if abs(med) <= max_abs_deg else 0.0


# --------------------------------------------------------------------------
# reading order
# --------------------------------------------------------------------------

@dataclass
class Line:
    """One text line: member box indices ordered left-to-right, plus the
    line's vertical band [top, bottom] in deskewed coordinates."""
    members: list[int]
    top: float
    bottom: float
    chunks: list[list[int]] = field(default_factory=list)

    @property
    def height(self) -> float:
        return self.bottom - self.top

    @property
    def center_y(self) -> float:
        return (self.top + self.bottom) / 2.0


def _band(quad_d: np.ndarray) -> tuple[float, float]:
    """Vertical band of a deskewed quad: mean of the 2 upper / 2 lower ys.
    Using means (not min/max) makes the band robust to one stray corner."""
    ys = np.sort(np.asarray(quad_d)[:, 1])
    return float(ys[:2].mean()), float(ys[2:].mean())


def split_columns(quads_d: list[np.ndarray], indices: list[int],
                  gap_factor: float = 2.0, min_side: int = 3,
                  min_y_coverage: float = 0.55, _depth: int = 0) -> list[list[int]]:
    """Conservative column split on deskewed quads. Returns groups of indices
    ordered left-to-right; [indices] unchanged when no confident cut exists.

    A cut requires: an x-projection gap (no box crosses it at any y) wider
    than gap_factor * median box height, at least min_side boxes per side,
    and each side spanning >= min_y_coverage of the block's height — so a
    single line with a big word gap can never trigger a phantom column.
    """
    if _depth >= 2 or len(indices) < 2 * min_side:
        return [indices]

    heights = [_band(quads_d[i])[1] - _band(quads_d[i])[0] for i in indices]
    med_h = float(np.median(heights))
    if med_h <= 0:
        return [indices]

    # merge the boxes' x-intervals; gaps between merged intervals are
    # x-ranges no box touches
    spans = sorted((float(quads_d[i][:, 0].min()), float(quads_d[i][:, 0].max()))
                   for i in indices)
    merged = [list(spans[0])]
    tol = 0.25 * med_h
    for lo, hi in spans[1:]:
        if lo <= merged[-1][1] + tol:
            merged[-1][1] = max(merged[-1][1], hi)
        else:
            merged.append([lo, hi])
    if len(merged) < 2:
        return [indices]

    gaps = [(merged[k + 1][0] - merged[k][1], (merged[k][1] + merged[k + 1][0]) / 2.0)
            for k in range(len(merged) - 1)]
    gaps = [g for g in gaps if g[0] >= gap_factor * med_h]
    if not gaps:
        return [indices]
    cut_x = max(gaps)[1]  # widest qualifying gap

    centers_x = {i: float(quads_d[i][:, 0].mean()) for i in indices}
    left = [i for i in indices if centers_x[i] < cut_x]
    right = [i for i in indices if centers_x[i] >= cut_x]
    if len(left) < min_side or len(right) < min_side:
        return [indices]

    def y_extent(group):
        bands = [_band(quads_d[i]) for i in group]
        return max(b for _, b in bands) - min(t for t, _ in bands)

    total = y_extent(indices)
    if total <= 0 or y_extent(left) < min_y_coverage * total \
            or y_extent(right) < min_y_coverage * total:
        return [indices]

    return (split_columns(quads_d, left, gap_factor, min_side, min_y_coverage, _depth + 1)
            + split_columns(quads_d, right, gap_factor, min_side, min_y_coverage, _depth + 1))


def cluster_lines(quads_d: list[np.ndarray], indices: list[int],
                  overlap_thresh: float = 0.4) -> list[Line]:
    """Group deskewed boxes into text lines by vertical-band overlap.

    A box joins the existing line it overlaps most, provided
    intersection / min(box_height, line_height) >= overlap_thresh.
    Normalizing by the smaller height keeps boxes with deep descenders or
    tall capitals (height outliers) attached to their true line.
    """
    items = []
    for i in indices:
        top, bottom = _band(quads_d[i])
        items.append((i, top, bottom))
    items.sort(key=lambda t: ((t[1] + t[2]) / 2.0, t[0]))

    lines: list[Line] = []
    for i, top, bottom in items:
        h = max(bottom - top, 1e-6)
        best, best_ov = None, overlap_thresh
        for line in lines:
            inter = min(bottom, line.bottom) - max(top, line.top)
            ov = inter / max(min(h, line.height), 1e-6)
            if ov >= best_ov:
                best, best_ov = line, ov
        if best is None:
            lines.append(Line(members=[i], top=top, bottom=bottom))
        else:
            n = len(best.members) + 1
            best.members.append(i)
            best.top += (top - best.top) / n        # running mean of bands
            best.bottom += (bottom - best.bottom) / n
    return lines


def reading_order(quads, column_split: bool = True,
                  overlap_thresh: float = 0.4,
                  gap_factor: float = 2.0) -> tuple[list[Line], float]:
    """Full reading order. Returns (lines, skew_deg); lines are in reading
    order, each line's members ordered left-to-right. Indices refer to the
    input quad list."""
    quads = [np.asarray(q, dtype=np.float64).reshape(4, 2) for q in quads]
    if not quads:
        return [], 0.0

    ordered = [order_points(q) for q in quads]
    theta = estimate_skew(ordered)
    deskewed = [rotate_points(q, -theta) for q in ordered]

    all_idx = list(range(len(quads)))
    groups = split_columns(deskewed, all_idx, gap_factor=gap_factor) \
        if column_split else [all_idx]

    result: list[Line] = []
    for group in groups:  # groups arrive left-to-right; read a column fully first
        lines = cluster_lines(deskewed, group, overlap_thresh=overlap_thresh)
        lines.sort(key=lambda l: l.center_y)
        for line in lines:
            line.members.sort(key=lambda i: (float(deskewed[i][:, 0].mean()), i))
        result.extend(lines)
    return result, theta


def chunk_line(line: Line, quads_d: list[np.ndarray],
               aspect_cap: float = 16.0) -> list[list[int]]:
    """Split a line's members (already x-ordered) into chunks whose merged
    width/height stays under aspect_cap.

    TrOCR resizes every crop to a 384x384 square; beyond ~16:1 the glyphs
    get squeezed into unreadability, so very long lines must be recognized
    in pieces.
    """
    chunks: list[list[int]] = []
    cur: list[int] = []
    cur_lo = cur_hi = 0.0
    h = max(line.height, 1e-6)
    for i in line.members:
        lo, hi = float(quads_d[i][:, 0].min()), float(quads_d[i][:, 0].max())
        if not cur:
            cur, cur_lo, cur_hi = [i], lo, hi
            continue
        new_lo, new_hi = min(cur_lo, lo), max(cur_hi, hi)
        if (new_hi - new_lo) / h > aspect_cap:
            chunks.append(cur)
            cur, cur_lo, cur_hi = [i], lo, hi
        else:
            cur, cur_lo, cur_hi = cur + [i], new_lo, new_hi
    if cur:
        chunks.append(cur)
    line.chunks = chunks
    return chunks


# --------------------------------------------------------------------------
# cropping
# --------------------------------------------------------------------------

def merge_quads(quads: list[np.ndarray]) -> np.ndarray:
    """Smallest rotated rectangle covering all quads, as an ordered quad
    (original image space)."""
    pts = np.vstack([np.asarray(q, dtype=np.float64).reshape(-1, 2) for q in quads])
    rect = cv2.minAreaRect(pts.astype(np.float32))
    return order_points(cv2.boxPoints(rect))


def expand_quad(quad: np.ndarray, pad_frac: float = 0.05) -> np.ndarray:
    """Push the ordered quad's corners outward along its own edge directions
    by pad_frac * height. No clipping to the image: the crop uses
    BORDER_REPLICATE, so slightly out-of-image corners are harmless and the
    quad stays a true parallelogram-ish shape."""
    q = np.asarray(quad, dtype=np.float64).copy()
    _, h = quad_size(q)
    pad = pad_frac * max(h, 1.0)
    ux = (q[1] - q[0]) + (q[2] - q[3])
    uy = (q[3] - q[0]) + (q[2] - q[1])
    nx = ux / max(np.linalg.norm(ux), 1e-6)
    ny = uy / max(np.linalg.norm(uy), 1e-6)
    q[0] += -nx * pad - ny * pad
    q[1] += +nx * pad - ny * pad
    q[2] += +nx * pad + ny * pad
    q[3] += -nx * pad + ny * pad
    return q


def perspective_crop(img: np.ndarray, quad, pad_frac: float = 0.05,
                     allow_rot90: bool = False,
                     interp: int = cv2.INTER_LINEAR) -> np.ndarray | None:
    """Rectify one ordered quad into an axis-aligned crop.

    Returns None for degenerate boxes instead of crashing warpPerspective.
    allow_rot90 rotates strongly vertical crops (h >= 2w) upright — pass it
    only for boxes known not to be single tall characters like 'I'.

    pad_frac controls how far the quad is expanded before warping. TrOCR likes a
    small margin (default 0.05) for handwriting ascenders/descenders. PaddleOCR
    recognition models, however, are trained on TIGHT crops (the detector already
    expands each box via its unclip ratio), so pass pad_frac=0.0 and
    interp=cv2.INTER_CUBIC there to match PP-OCR's own `get_rotate_crop_image` —
    extra padding double-expands the box and pulls neighboring glyphs into the
    crop, which hurts recognition on dense pages.
    """
    q = (expand_quad(order_points(quad), pad_frac) if pad_frac else order_points(quad)
         ).astype(np.float32)
    w = int(round(max(np.linalg.norm(q[1] - q[0]), np.linalg.norm(q[2] - q[3]))))
    h = int(round(max(np.linalg.norm(q[3] - q[0]), np.linalg.norm(q[2] - q[1]))))
    if w < 2 or h < 2:
        return None
    dst = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=np.float32)
    m = cv2.getPerspectiveTransform(q, dst)
    warped = cv2.warpPerspective(img, m, (w, h), flags=interp,
                                 borderMode=cv2.BORDER_REPLICATE)
    if allow_rot90 and h >= 2 * w:
        # np.rot90 returns a negative-stride VIEW; torch.from_numpy (in the
        # TrOCR processor) rejects those, so force a contiguous copy.
        warped = np.ascontiguousarray(np.rot90(warped))  # CCW, matching PaddleOCR
    return warped


# --------------------------------------------------------------------------
# output assembly / debug rendering
# --------------------------------------------------------------------------

def assemble_text(lines: list[Line], line_texts: list[str],
                  para_gap_factor: float = 1.0,
                  column_breaks: set[int] | None = None) -> str:
    """Join per-line texts in reading order. Inserts a blank line when the
    vertical gap to the previous line exceeds para_gap_factor * median line
    height (a paragraph break), or at a column boundary."""
    assert len(lines) == len(line_texts)
    if not lines:
        return ""
    med_h = float(np.median([max(l.height, 1e-6) for l in lines]))
    out: list[str] = []
    for k, (line, text) in enumerate(zip(lines, line_texts)):
        if k > 0:
            gap = line.top - lines[k - 1].bottom
            if (column_breaks and k in column_breaks) or gap > para_gap_factor * med_h:
                out.append("")
        out.append(text)
    return "\n".join(out)


def annotate(img_rgb: np.ndarray, quads: list[np.ndarray],
             lines: list[Line]) -> np.ndarray:
    """Debug overlay: each box outlined and numbered with its reading order,
    colored per line. Makes ordering mistakes visible at a glance."""
    out = img_rgb.copy()
    palette = [(46, 204, 113), (52, 152, 219), (231, 76, 60), (241, 196, 15),
               (155, 89, 182), (26, 188, 156), (230, 126, 34), (149, 165, 166)]
    n = 0
    scale = max(out.shape[0], out.shape[1]) / 1500.0
    fs = max(0.5, 0.9 * scale)
    th = max(1, int(round(2 * scale)))
    for li, line in enumerate(lines):
        color = palette[li % len(palette)]
        for i in line.members:
            n += 1
            q = np.asarray(quads[i], dtype=np.int32).reshape(-1, 2)
            cv2.polylines(out, [q], isClosed=True, color=color, thickness=th)
            pos = (int(q[:, 0].min()), max(int(q[:, 1].min()) - 4, 12))
            cv2.putText(out, str(n), pos, cv2.FONT_HERSHEY_SIMPLEX, fs,
                        color, th, cv2.LINE_AA)
    return out


_PALETTE = [(46, 204, 113), (52, 152, 219), (231, 76, 60), (241, 196, 15),
            (155, 89, 182), (26, 188, 156), (230, 126, 34), (192, 57, 43)]


def _load_font(size: int):
    from PIL import ImageFont
    for name in ("DejaVuSans.ttf",
                 "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                 "arial.ttf", r"C:\Windows\Fonts\arial.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except Exception:
            continue
    try:
        return ImageFont.load_default(size)  # Pillow >= 10
    except TypeError:
        return ImageFont.load_default()


def _wrap(draw, text: str, font, max_w: float) -> list[str]:
    words = text.split()
    if not words:
        return [""]
    out, cur = [], words[0]
    for w in words[1:]:
        if draw.textlength(cur + " " + w, font=font) <= max_w:
            cur += " " + w
        else:
            out.append(cur)
            cur = w
    out.append(cur)
    return out


def compose_transcript(img_rgb: np.ndarray, line_boxes: list[list[np.ndarray]],
                       line_texts: list[str], panel_frac: float = 0.6) -> np.ndarray:
    """One image: the page with per-line numbered, color-coded boxes on the
    left, and the matching numbered transcript on the right.

    line_boxes[k] are the detected quads of line k; line_texts[k] is its text.
    The number and color on each box match its transcript entry, so you can
    read the page and the recognized text side by side and immediately see
    any mismatch. Pure numpy/PIL/cv2 — no models involved."""
    from PIL import Image, ImageDraw

    h, w = img_rgb.shape[:2]
    scale = max(h, w) / 1500.0
    th = max(1, int(round(2 * scale)))
    fs_cv = max(0.5, 1.0 * scale)

    # --- left: draw boxes, label each line at its leftmost box ---
    left = img_rgb.copy()
    for li, boxes in enumerate(line_boxes):
        color = _PALETTE[li % len(_PALETTE)]
        anchor = None
        for box in boxes:
            q = np.asarray(box, dtype=np.int32).reshape(-1, 2)
            cv2.polylines(left, [q], isClosed=True, color=color, thickness=th)
            x0, y0 = int(q[:, 0].min()), int(q[:, 1].min())
            if anchor is None or x0 < anchor[0]:
                anchor = (x0, y0)
        if anchor is not None:
            cv2.putText(left, str(li + 1), (anchor[0], max(anchor[1] - 6, 14)),
                        cv2.FONT_HERSHEY_SIMPLEX, fs_cv, color, th, cv2.LINE_AA)

    # --- right: numbered transcript panel ---
    panel_w = max(360, int(w * panel_frac))
    fsize = max(16, int(round(0.020 * h)))
    line_h = int(fsize * 1.35)
    pad = fsize
    font = _load_font(fsize)
    num_font = _load_font(fsize)

    probe = ImageDraw.Draw(Image.new("RGB", (8, 8)))
    indent = int(max((probe.textlength(f"{k + 1}.", font=num_font)
                      for k in range(max(len(line_texts), 1))), default=0) + 0.6 * fsize)
    text_w = max(panel_w - 2 * pad - indent, 8 * fsize)

    wrapped = []
    for li, txt in enumerate(line_texts):
        segs = _wrap(probe, txt if txt.strip() else "(blank)", font, text_w)
        wrapped.append((li, _PALETTE[li % len(_PALETTE)], segs))

    panel_h = pad + sum(len(segs) * line_h + int(0.5 * line_h)
                        for _, _, segs in wrapped) + pad
    canvas_h = max(h, panel_h)

    panel = Image.new("RGB", (panel_w, canvas_h), (255, 255, 255))
    pd = ImageDraw.Draw(panel)
    y = pad
    for li, color, segs in wrapped:
        pd.text((pad, y), f"{li + 1}.", font=num_font, fill=color)
        for seg in segs:
            pd.text((pad + indent, y), seg, font=font, fill=(20, 20, 25))
            y += line_h
        y += int(0.5 * line_h)

    left_canvas = np.full((canvas_h, w, 3), 255, dtype=np.uint8)
    left_canvas[:h, :w] = left
    sep = np.full((canvas_h, max(2, th), 3), 220, dtype=np.uint8)
    return np.concatenate([left_canvas, sep, np.asarray(panel)], axis=1)


In [ ]:
%%writefile detector.py
"""PaddleOCR text-detection wrapper → clean (N,4,2) quads in image coordinates.

Paddle-only (no torch/transformers), so it is shared by *both* pipelines — the
handwriting engine (`ocr_engine.py`, TrOCR recognition) and the multilingual
engine (`multilingual/ml_engine.py`, PP-OCR recognition) — without either
dragging in the other's recognizer.
"""

from __future__ import annotations

import os
import sys

import cv2
import numpy as np

import pipeline

DET_MODEL_NAME = os.environ.get("OCR_DET_MODEL", "PP-OCRv5_server_det")
MIN_DET_SCORE = 0.30


class Detector:
    """PP-OCRv5 text detection wrapper that returns clean (N,4,2) quads in
    original-image coordinates."""

    def __init__(self, model_name: str = DET_MODEL_NAME,
                 enable_mkldnn: bool | None = None):
        from paddleocr import TextDetection  # lazy: paddle is optional
        self._cls = TextDetection
        self._model_name = model_name
        if enable_mkldnn is None:
            env = os.environ.get("OCR_DET_MKLDNN")
            # paddle's PIR + oneDNN executor is broken on Windows CPU
            # (ConvertPirAttribute2RuntimeAttribute NotImplementedError)
            enable_mkldnn = env == "1" if env is not None else sys.platform != "win32"
        self._det = TextDetection(model_name=model_name, enable_mkldnn=enable_mkldnn)

    def __call__(self, img_rgb: np.ndarray, min_score: float = MIN_DET_SCORE) -> np.ndarray:
        # PaddleOCR consumes cv2-style BGR; gradio/PIL hand us RGB.
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        try:
            results = self._det.predict(img_bgr, batch_size=1)
        except NotImplementedError as e:  # oneDNN executor bug -> retry without it
            if "Pir" not in str(e) and "onednn" not in str(e).lower():
                raise
            self._det = self._cls(model_name=self._model_name, enable_mkldnn=False)
            results = self._det.predict(img_bgr, batch_size=1)

        quads: list[np.ndarray] = []
        for res in results:
            polys = res.get("dt_polys", None) if hasattr(res, "get") else None
            if polys is None:
                continue
            scores = res.get("dt_scores", None)
            for k, poly in enumerate(polys):
                score = float(scores[k]) if scores is not None and k < len(scores) else 1.0
                if score < min_score:
                    continue
                arr = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
                if arr.shape[0] > 4:  # polygon mode -> reduce to min-area quad
                    arr = pipeline.merge_quads([arr])
                if arr.shape[0] != 4:
                    continue
                w, h = pipeline.quad_size(pipeline.order_points(arr))
                if w < 2 or h < 2:
                    continue
                quads.append(arr)
        return np.array(quads, dtype=np.float64) if quads else np.zeros((0, 4, 2))


In [ ]:
%%writefile evaluate.py
"""OCR evaluation harness: CER / WER + an order-free recognition diagnostic,
plus baseline runners so several systems are scored on the *same* data.

Metrics
-------
- **CER** (Character Error Rate) = Levenshtein(ref, hyp) / len(ref), aggregated
  at corpus level (sum of edits / sum of ref chars — the standard way). Lower
  is better; can exceed 100% when a system hallucinates. This is the headline.
- **WER** (Word Error Rate) = same, on whitespace tokens.
- **WordAcc** = order-free word accuracy = fraction of reference words present
  in the prediction regardless of position (multiset intersection / n_ref).
  Higher is better. This separates recognition from reading order WITHOUT the
  paradoxes of a char-level "order-invariant CER": sorting words/chars to remove
  order misaligns recognition errors and can exceed the document CER, so such a
  decomposition is ill-posed on real data. WordAcc is a clean, bounded [0,1]
  signal — high WordAcc with high CER => ordering/segmentation issue; low
  WordAcc => genuine recognition issue. It is NOT subtracted from CER.

A "system" is just a function image_rgb -> text (and optionally -> list of
lines, for the order diagnostic). Their pipeline, Tesseract, EasyOCR and
PaddleOCR are all wired up below behind lazy imports.

CLI:
    # your own labeled folder
    python evaluate.py --data DIR
    python evaluate.py --data DIR --baselines tesseract,easyocr,paddleocr --csv out.csv
    # a public dataset (Hugging Face) — IAM test lines, quick 200-sample check:
    python evaluate.py --hf iam-lines --n 200
    python evaluate.py --hf iam-lines            # full IAM test (2,920 lines)

--data layout: each image `foo.jpg` has its ground-truth transcript in a sibling
text file `foo.txt` (or `foo.gt.txt`), one physical line per line, in reading
order. Feed page images for document-level scores, or single-line crops for
line-level scores.

--hf: a preset in HF_DATASETS (currently 'iam-lines' = Teklia/IAM-line) or any
hub dataset id with an image column + a text column (auto-detected). Line-level
datasets are scored on the recognizer alone (no detection/ordering), directly
comparable to published TrOCR CER (~2.9% on IAM test).
"""

from __future__ import annotations

import argparse
import re
import string
import time
from dataclasses import dataclass, field
from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


# --------------------------------------------------------------------------
# edit distance + normalization
# --------------------------------------------------------------------------

def levenshtein(a, b) -> int:
    """Edit distance between two sequences (strings for CER, token lists for WER)."""
    if a == b:
        return 0
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        ai = a[i - 1]
        for j in range(1, lb + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[lb]


def normalize(s: str, lowercase: bool = False, strip_punct: bool = False,
              collapse_ws: bool = True) -> str:
    """Text normalization applied to BOTH ref and hyp before scoring.

    Default collapses runs of whitespace (incl. newlines) to single spaces so
    detection/segmentation formatting differences don't inflate CER, while
    leaving case and punctuation intact (standard CER). Flags let you relax it.
    """
    if lowercase:
        s = s.lower()
    if strip_punct:
        s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s).strip() if collapse_ws else s.strip()
    return s


@dataclass
class NormCfg:
    lowercase: bool = False
    strip_punct: bool = False
    collapse_ws: bool = True

    def __call__(self, s: str) -> str:
        return normalize(s, self.lowercase, self.strip_punct, self.collapse_ws)


def cer_counts(ref: str, hyp: str, norm: NormCfg) -> tuple[int, int]:
    r, h = norm(ref), norm(hyp)
    return levenshtein(r, h), len(r)


def wer_counts(ref: str, hyp: str, norm: NormCfg) -> tuple[int, int]:
    r, h = norm(ref).split(), norm(hyp).split()
    return levenshtein(r, h), len(r)


def word_acc_counts(ref: str, hyp: str, norm: NormCfg) -> tuple[int, int]:
    """Order-free recognition signal: (matched, n_ref_words) where `matched` is
    the multiset intersection of reference and predicted words.

    word accuracy = matched / n_ref_words = fraction of ground-truth words the
    system produced *regardless of position*. This isolates recognition quality
    from reading order without the paradoxes of a char-level "order-invariant
    CER" (sorting characters/words misaligns recognition errors and can exceed
    the document CER). High word-accuracy with high CER => ordering/segmentation
    problem; low word-accuracy => genuine recognition problem."""
    from collections import Counter
    r = Counter(norm(ref).split())
    h = Counter(norm(hyp).split())
    matched = sum((r & h).values())
    return matched, sum(r.values())


# --------------------------------------------------------------------------
# dataset
# --------------------------------------------------------------------------

@dataclass
class Sample:
    name: str
    gt: str
    image_path: Path | None = None
    image: object | None = None  # in-memory PIL.Image or RGB np.ndarray (optional)

    def image_rgb(self):
        import cv2
        import numpy as np
        if self.image is not None:
            arr = np.asarray(self.image)
            if arr.ndim == 2:
                arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
            elif arr.shape[2] == 4:
                arr = cv2.cvtColor(arr, cv2.COLOR_RGBA2RGB)
            return np.ascontiguousarray(arr)
        bgr = cv2.imdecode(np.fromfile(self.image_path, np.uint8), cv2.IMREAD_COLOR)
        if bgr is None:
            raise ValueError(f"could not decode {self.image_path}")
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def load_pairs(folder: str | Path) -> list[Sample]:
    """Local folder: each image has a sibling .txt / .gt.txt ground truth."""
    folder = Path(folder)
    samples: list[Sample] = []
    for img in sorted(folder.rglob("*")):
        if img.suffix.lower() not in IMAGE_EXTS:
            continue
        if any(t in img.stem for t in ("_panel", "_overlay")):  # skip our outputs
            continue
        gt_path = None
        for cand in (img.with_suffix(".gt.txt"), img.with_suffix(".txt"),
                     img.with_name(img.stem + ".gt.txt")):
            if cand.is_file():
                gt_path = cand
                break
        if gt_path is None:
            print(f"  (skip {img.name}: no ground-truth .txt beside it)")
            continue
        samples.append(Sample(name=img.name, image_path=img,
                              gt=gt_path.read_text(encoding="utf-8")))
    return samples


# verified-good presets; `level='line'` -> score the recognizer directly,
# `level='page'` -> score the full detection+ordering+recognition pipeline.
HF_DATASETS = {
    "iam-lines": dict(path="Teklia/IAM-line", image_col="image", text_col="text",
                      split="test", level="line",
                      note="IAM test lines; compare to TrOCR-large CER ~2.9%"),
    "iam-sentences": dict(path="alpayariyak/IAM_Sentences", image_col="image",
                          text_col="text", split="train", level="page",
                          note="IAM sentences as MULTI-LINE images; tests the full "
                               "pipeline. Recognition CER is optimistic (TrOCR trained "
                               "on IAM); detection+ordering test is valid."),
}

_IMG_COL_HINTS = ("image", "img", "im", "picture")
_TXT_COL_HINTS = ("text", "label", "transcription", "transcript", "sentence",
                  "gt", "ground_truth", "caption")


def _auto_col(row: dict, hints) -> str:
    keys = list(row.keys())
    for h in hints:
        for k in keys:
            if k.lower() == h:
                return k
    for h in hints:
        for k in keys:
            if h in k.lower():
                return k
    raise KeyError(f"could not auto-detect column from {keys} (hints={hints}); "
                   f"pass image_col=/text_col= explicitly")


def load_hf(name_or_path: str, split: str | None = None, n: int | None = None,
            image_col: str | None = None, text_col: str | None = None,
            config: str | None = None, streaming: bool = True) -> list[Sample]:
    """Load image+text pairs from a Hugging Face dataset.

    `name_or_path` may be a preset key in HF_DATASETS (e.g. 'iam-lines') or any
    hub dataset id exposing an image column and a text column. `n` caps the
    number of samples (streaming, so it won't download the whole set for a
    quick check). Columns and split are auto-filled from the preset / detected.
    """
    from datasets import load_dataset

    preset = HF_DATASETS.get(name_or_path, {})
    path = preset.get("path", name_or_path)
    image_col = image_col or preset.get("image_col")
    text_col = text_col or preset.get("text_col")
    config = config or preset.get("config")
    split = split or preset.get("split", "test")

    ds = load_dataset(path, name=config, split=split, streaming=streaming)
    samples: list[Sample] = []
    for i, row in enumerate(ds):
        if n is not None and i >= n:
            break
        ic = image_col or _auto_col(row, _IMG_COL_HINTS)
        tc = text_col or _auto_col(row, _TXT_COL_HINTS)
        samples.append(Sample(name=f"{path}:{split}:{i}", gt=str(row[tc]),
                              image=row[ic]))
    return samples


def is_line_level(name_or_path: str) -> bool:
    return HF_DATASETS.get(name_or_path, {}).get("level") == "line"


def build_iam_pages(n_pages: int = 20, lines_per_page: tuple[int, int] = (4, 8),
                    max_skew_deg: float = 3.0, gap_px: int = 22, margin_px: int = 50,
                    max_indent_px: int = 60, seed: int = 0,
                    source_split: str = "test") -> list[Sample]:
    """Synthesize multi-line 'pages' by stacking real IAM line images, with
    random per-line indent and a small page skew. The skew makes naive
    top-to-bottom ordering fail, so a low CER here means `pipeline.reading_order`
    survived; WordAcc stays high since the stacking doesn't change recognition.

    Recognition CER is still IAM-optimistic (TrOCR trained on IAM); the point of
    this set is to measure detection + line clustering + ordering."""
    import cv2
    import numpy as np
    from datasets import load_dataset

    rng = np.random.default_rng(seed)
    it = iter(load_dataset("Teklia/IAM-line", split=source_split, streaming=True))
    pages: list[Sample] = []
    for p in range(n_pages):
        k = int(rng.integers(lines_per_page[0], lines_per_page[1] + 1))
        rows, texts = [], []
        for _ in range(k):
            row = next(it)
            rows.append(np.asarray(row["image"].convert("L")))
            texts.append(row["text"])
        indents = [int(rng.integers(0, max_indent_px + 1)) for _ in rows]
        width = margin_px * 2 + max(r.shape[1] + indents[i] for i, r in enumerate(rows))
        height = margin_px * 2 + sum(r.shape[0] for r in rows) + gap_px * (k - 1)
        canvas = np.full((height, width), 255, np.uint8)
        y = margin_px
        for r, ind in zip(rows, indents):
            x = margin_px + ind
            canvas[y:y + r.shape[0], x:x + r.shape[1]] = r
            y += r.shape[0] + gap_px
        if max_skew_deg:
            ang = float(rng.uniform(-max_skew_deg, max_skew_deg))
            m = cv2.getRotationMatrix2D((width / 2, height / 2), ang, 1.0)
            canvas = cv2.warpAffine(canvas, m, (width, height),
                                    borderValue=255, flags=cv2.INTER_LINEAR)
        pages.append(Sample(name=f"iam-page-{p:03d}", gt="\n".join(texts),
                            image=cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)))
    return pages


def _gnhk_words(data) -> list[dict]:
    """GNHK json is normally a list of word dicts; tolerate a dict wrapper."""
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for v in data.values():
            if isinstance(v, list):
                return v
    return []


def _gnhk_reading_order_text(words: list[dict], drop_special: bool = True,
                             keep_types: set[str] | None = None) -> str:
    """Reconstruct a page's reading-order transcript from GNHK word annotations
    (each: 'text', 'polygon' {x0,y0..x3,y3}, 'line_idx', 'type'). Groups by
    line_idx, orders lines by mean-y, words within a line by x.

    `%...%` tokens (%NA%/%math%/%SC%/...) mark non-transcribable content and are
    dropped by default — same as the GNHK recognition benchmark. `keep_types`
    optionally restricts to certain region types (e.g. {'H'} for handwritten),
    once you've seen the type codes in the real data."""
    lines: dict[int, list[tuple[float, float, str]]] = {}
    for w in words:
        t = str(w.get("text", "")).strip()
        if not t or (drop_special and t.startswith("%") and t.endswith("%")):
            continue
        if keep_types is not None and w.get("type") not in keep_types:
            continue
        poly = w.get("polygon", {}) or {}
        xs = [poly.get(f"x{i}", 0) for i in range(4)]
        ys = [poly.get(f"y{i}", 0) for i in range(4)]
        lines.setdefault(int(w.get("line_idx", -1)), []).append(
            (min(xs), min(ys), t))
    ordered = sorted(lines.values(),
                     key=lambda ws: sum(y for _, y, _ in ws) / len(ws))
    return "\n".join(" ".join(t for _, _, t in sorted(ws, key=lambda e: e[0]))
                     for ws in ordered)


def load_gnhk(folder: str | Path, drop_special: bool = True,
              keep_types: set[str] | None = None) -> list[Sample]:
    """Load GNHK (real-photo handwriting) pages from a locally downloaded folder
    (https://goodnotes.com/gnhk — agree to the CC-BY-4.0 terms, then unzip the
    train/test zips here). Expects image files with sibling .json word
    annotations; GT is reconstructed in reading order from line_idx."""
    import json
    folder = Path(folder)
    samples: list[Sample] = []
    for jf in sorted(folder.rglob("*.json")):
        img = next((c for ext in IMAGE_EXTS
                    for c in [jf.with_suffix(ext)] if c.is_file()), None)
        if img is None:
            continue
        try:
            words = _gnhk_words(json.loads(jf.read_text(encoding="utf-8")))
        except json.JSONDecodeError:
            continue
        gt = _gnhk_reading_order_text(words, drop_special, keep_types)
        if gt.strip():
            samples.append(Sample(name=jf.stem, image_path=img, gt=gt))
    return samples


# --------------------------------------------------------------------------
# scoring
# --------------------------------------------------------------------------

@dataclass
class Score:
    system: str
    n: int = 0
    cer_edits: int = 0
    cer_ref: int = 0
    wer_edits: int = 0
    wer_ref: int = 0
    wa_matched: int = 0
    wa_total: int = 0
    seconds: float = 0.0
    per_item: list[dict] = field(default_factory=list)

    @property
    def cer(self) -> float:
        return self.cer_edits / max(self.cer_ref, 1)

    @property
    def wer(self) -> float:
        return self.wer_edits / max(self.wer_ref, 1)

    @property
    def word_acc(self) -> float | None:
        # order-free fraction of reference words recovered (higher is better)
        return self.wa_matched / self.wa_total if self.wa_total else None

    @property
    def sec_per_img(self) -> float:
        return self.seconds / max(self.n, 1)


def evaluate_system(name: str, samples: list[Sample], predict_text,
                    predict_lines=None, norm: NormCfg | None = None,
                    progress: bool = False, word_metrics: bool = True) -> Score:
    """Score one system. predict_text(image_rgb)->str. WordAcc is computed
    order-free from the text, so predict_lines is not required (kept for
    backward compatibility, ignored).

    word_metrics=False skips WER and WordAcc — set it for **space-free CJK**,
    where there are no word boundaries: stripping spaces makes the whole page one
    token, so WER collapses to 100% and WordAcc to 0% (meaningless). Only CER
    (character-level) is valid there; the table then shows WER/WordAcc as '—'.

    progress=True prints a line per image (running CER + per-image seconds) so a
    slow run — e.g. heavy PaddleOCR models on a CPU runtime — shows it is working,
    not hung."""
    norm = norm or NormCfg()
    sc = Score(system=name)
    total = len(samples)
    for s in samples:
        img = s.image_rgb()
        t0 = time.perf_counter()
        hyp = predict_text(img)
        dt = time.perf_counter() - t0

        ce, cr = cer_counts(s.gt, hyp, norm)
        sc.n += 1
        sc.cer_edits += ce; sc.cer_ref += cr
        sc.seconds += dt
        item = {"system": name, "image": s.name, "cer": ce / max(cr, 1),
                "ref_chars": cr, "seconds": round(dt, 3)}
        if word_metrics:
            we, wr = wer_counts(s.gt, hyp, norm)
            wm, wt = word_acc_counts(s.gt, hyp, norm)
            sc.wer_edits += we; sc.wer_ref += wr
            sc.wa_matched += wm; sc.wa_total += wt
            item.update(wer=we / max(wr, 1), word_acc=wm / max(wt, 1))
        sc.per_item.append(item)
        if progress:
            print(f"  [{name}] {sc.n}/{total} {s.name}: cer={sc.cer:.3f} "
                  f"({dt:.1f}s)", flush=True)
    return sc


# --------------------------------------------------------------------------
# systems under test (lazy imports — none required unless used)
# --------------------------------------------------------------------------

def recognizer_predictor(recognizer=None, **rec_kwargs):
    """predict_text using ONLY the TrOCR recognizer (no detection/ordering).
    Use on single-line images (e.g. IAM lines) for a recognizer-quality number
    directly comparable to published TrOCR CER."""
    if recognizer is None:
        from ocr_engine import Recognizer
        recognizer = Recognizer(**rec_kwargs)
    return lambda img: recognizer([img])[0]


def evaluate_recognizer(name: str, samples: list[Sample], recognizer=None,
                        norm: NormCfg | None = None, batch_size: int = 16) -> Score:
    """Fast batched recognizer-only scoring for line datasets — runs the whole
    set through TrOCR in batches instead of one image at a time."""
    norm = norm or NormCfg()
    if recognizer is None:
        from ocr_engine import Recognizer
        recognizer = Recognizer()
    imgs = [s.image_rgb() for s in samples]
    t0 = time.perf_counter()
    hyps: list[str] = []
    for i in range(0, len(imgs), batch_size):
        hyps.extend(recognizer(imgs[i:i + batch_size], batch_size=batch_size))
    dt = time.perf_counter() - t0

    sc = Score(system=name, seconds=dt)
    for s, hyp in zip(samples, hyps):
        ce, cr = cer_counts(s.gt, hyp, norm)
        we, wr = wer_counts(s.gt, hyp, norm)
        wm, wt = word_acc_counts(s.gt, hyp, norm)
        sc.n += 1
        sc.cer_edits += ce; sc.cer_ref += cr
        sc.wer_edits += we; sc.wer_ref += wr
        sc.wa_matched += wm; sc.wa_total += wt
        sc.per_item.append({"system": name, "image": s.name, "cer": ce / max(cr, 1),
                            "wer": we / max(wr, 1), "word_acc": wm / max(wt, 1),
                            "ref_chars": cr})
    return sc


def pipeline_predictors(engine=None, **engine_kwargs):
    """Return (predict_text, predict_lines) for THIS project's pipeline."""
    if engine is None:
        from ocr_engine import OcrEngine
        engine = OcrEngine(**engine_kwargs)
    cache: dict[int, dict] = {}

    def _run(img):
        key = id(img)
        if key not in cache:
            cache.clear()
            cache[key] = engine.run(img)
        return cache[key]

    return (lambda img: _run(img)["text"],
            lambda img: [l["text"] for l in _run(img)["lines"]])


def baseline_predict(name: str):
    """Return predict_text for a baseline OCR system (lazy, raises if missing)."""
    name = name.lower()
    if name == "tesseract":
        import pytesseract
        from PIL import Image
        return lambda img: pytesseract.image_to_string(Image.fromarray(img))
    if name == "easyocr":
        import easyocr
        reader = easyocr.Reader(["en"], gpu=True)
        return lambda img: "\n".join(reader.readtext(img, detail=0, paragraph=True))
    if name == "paddleocr":
        import cv2
        from paddleocr import PaddleOCR
        # enable_mkldnn=False avoids paddle's PIR+oneDNN crash
        # (ConvertPirAttribute2RuntimeAttribute), same fix Detector uses.
        ocr = None
        for kw in (dict(lang="en", enable_mkldnn=False, use_doc_orientation_classify=False,
                        use_doc_unwarping=False, use_textline_orientation=False),
                   dict(lang="en", enable_mkldnn=False),
                   dict(use_angle_cls=False, lang="en", enable_mkldnn=False),  # 2.x
                   dict(lang="en")):
            try:
                ocr = PaddleOCR(**kw)
                break
            except TypeError:
                continue
        if ocr is None:
            raise RuntimeError("could not initialize PaddleOCR")
        def _run(img):
            bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            if hasattr(ocr, "predict"):                      # 3.x: list of dict-like
                try:
                    out = []
                    for r in ocr.predict(bgr):
                        texts = r.get("rec_texts") if hasattr(r, "get") else None
                        if texts:
                            out.extend(texts)
                    return "\n".join(out)
                except Exception:
                    pass
            res = ocr.ocr(bgr)                               # 2.x: [[[box,(text,score)],...]]
            out = []
            for page in res or []:
                for line in page or []:
                    try:
                        out.append(line[1][0])
                    except (IndexError, TypeError):
                        pass
            return "\n".join(out)
        return _run
    raise ValueError(f"unknown baseline: {name}")


# --------------------------------------------------------------------------
# reporting
# --------------------------------------------------------------------------

def format_table(scores: list[Score]) -> str:
    # Standard OCR metrics: CER and WER (both corpus-level Levenshtein, lower is better).
    # WordAcc is kept in the CSV for per-image diagnostics but not shown in the table —
    # it is order-free and not a standard benchmark metric.
    head = f"{'system':<20}{'CER':>8}{'WER':>8}{'sec/img':>9}{'n':>5}"
    rows = [head, "-" * len(head)]
    for s in sorted(scores, key=lambda x: x.cer):
        wer = f"{s.wer:>8.1%}" if s.wer_ref else f"{'—':>8}"
        rows.append(f"{s.system:<20}{s.cer:>8.1%}{wer}{s.sec_per_img:>9.2f}{s.n:>5}")
    return "\n".join(rows)


def save_csv(scores: list[Score], path: str | Path) -> None:
    import csv
    rows = [it for s in scores for it in s.per_item]
    if not rows:
        return
    keys = ["system", "image", "cer", "wer", "word_acc", "ref_chars", "seconds"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    src = ap.add_mutually_exclusive_group(required=True)
    src.add_argument("--data", help="folder of images + sibling .txt ground truth")
    src.add_argument("--hf", help=f"HF dataset: preset ({', '.join(HF_DATASETS)}) or any hub id")
    ap.add_argument("--split", default="test", help="HF split (default: test)")
    ap.add_argument("--n", type=int, default=None, help="cap number of HF samples (quick check)")
    ap.add_argument("--config", default=None, help="HF dataset config/subset name")
    ap.add_argument("--mode", choices=["auto", "recognizer", "pipeline"], default="auto",
                    help="auto: recognizer for line datasets, full pipeline otherwise")
    ap.add_argument("--baselines", default="", help="comma list: tesseract,easyocr,paddleocr")
    ap.add_argument("--csv", default="eval_results.csv")
    ap.add_argument("--lowercase", action="store_true")
    ap.add_argument("--strip-punct", action="store_true")
    args = ap.parse_args()

    if args.hf:
        samples = load_hf(args.hf, split=args.split, n=args.n, config=args.config)
        line_level = is_line_level(args.hf)
        print(f"loaded {len(samples)} samples from HF {args.hf} [{args.split}]")
    else:
        samples = load_pairs(args.data)
        line_level = False
        print(f"loaded {len(samples)} samples from {args.data}")
    if not samples:
        print("No (image, ground-truth) pairs found. See module docstring for layout.")
        return 1

    norm = NormCfg(lowercase=args.lowercase, strip_punct=args.strip_punct)
    use_recognizer = args.mode == "recognizer" or (args.mode == "auto" and line_level)

    scores: list[Score] = []
    if use_recognizer:
        print("scoring recognizer only (no detection/ordering)")
        scores.append(evaluate_recognizer("trocr-recognizer", samples, norm=norm))
    else:
        pt, pl = pipeline_predictors()
        scores.append(evaluate_system("this-pipeline", samples, pt, pl, norm))

    for b in [x.strip() for x in args.baselines.split(",") if x.strip()]:
        try:
            scores.append(evaluate_system(b, samples, baseline_predict(b), norm=norm))
        except Exception as e:  # missing dep / runtime error in a baseline
            print(f"  (skip baseline {b}: {type(e).__name__}: {e})")

    print("\n" + format_table(scores))
    save_csv(scores, args.csv)
    print(f"\nper-image breakdown -> {args.csv}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile ml_engine.py
"""Multilingual full-page OCR: PP-OCRv5 detection -> reading-order
reconstruction (shared pipeline.py) -> PP-OCRv5 recognition.

Unlike the English pipeline (which recognizes with TrOCR), here *both* detection
and recognition are PaddleOCR PP-OCRv5 models, so it stays light and supports
every script PP-OCR ships. The contribution is the same deterministic
reading-order stage, inserted between PaddleOCR's detector and recognizer:

  * We run a strong detector (PP-OCRv5 *server* det) ourselves and recognize
    EVERY detected box. PaddleOCR's built-in pipeline silently drops boxes whose
    recognition confidence falls below an internal threshold, so words go
    missing from the output; recognizing each detected box keeps them.
  * Boxes are emitted in a geometry-derived reading order (deskew -> line
    cluster -> left-to-right within a line -> column split), instead of the
    detector's raw output order, so the transcript reads correctly.

Scope: left-to-right scripts (Latin, CJK, Indic). RTL (Arabic/Hebrew) is out of
scope — the ordering sorts left-to-right within each line.

CJK note: words inside a line are joined with no separator for CJK languages
(Chinese/Japanese/Korean have no inter-word spaces) and with a space otherwise,
so character-level CER is not inflated by spurious spaces.
"""
from __future__ import annotations

import sys
import time
from pathlib import Path

import cv2
import numpy as np

# make `import pipeline` / `import ocr_engine` work when run from this subfolder
_ROOT = str(Path(__file__).resolve().parent.parent)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

import pipeline
from detector import Detector  # paddle-only detection wrapper (no TrOCR dependency)

# PP-OCRv5 server rec covers Chinese + English + Japanese (+ pinyin) in one
# model. Other scripts use their own PP-OCR language recognition model. These
# names are resolved at load time; PaddleRecognizer falls back gracefully if a
# given name is unavailable in the installed PaddleOCR version.
DEFAULT_REC_MODEL = "PP-OCRv5_server_rec"
REC_MODELS = {
    "ch": "PP-OCRv5_server_rec", "chinese": "PP-OCRv5_server_rec",
    "chinese_cht": "chinese_cht_PP-OCRv5_mobile_rec",
    "en": "PP-OCRv5_server_rec",
    "japan": "PP-OCRv5_server_rec", "ja": "PP-OCRv5_server_rec",
    "korean": "korean_PP-OCRv5_mobile_rec", "ko": "korean_PP-OCRv5_mobile_rec",
    "latin": "latin_PP-OCRv5_mobile_rec",
    "fr": "latin_PP-OCRv5_mobile_rec", "de": "latin_PP-OCRv5_mobile_rec",
    "es": "latin_PP-OCRv5_mobile_rec", "it": "latin_PP-OCRv5_mobile_rec",
    "pt": "latin_PP-OCRv5_mobile_rec",
}
# languages written without inter-word spaces -> join boxes with no separator
_NO_SPACE_LANGS = {"ch", "chinese", "chinese_cht", "japan", "ja", "korean", "ko"}


def rec_model_for(lang: str) -> str:
    return REC_MODELS.get(lang, DEFAULT_REC_MODEL)


class PaddleRecognizer:
    """Wraps a PaddleOCR PP-OCRv5 text-recognition model over RGB crops."""

    def __init__(self, model_name: str = DEFAULT_REC_MODEL,
                 enable_mkldnn: bool | None = None):
        from paddleocr import TextRecognition
        if enable_mkldnn is None:
            enable_mkldnn = sys.platform != "win32"  # PIR+oneDNN broken on win CPU
        self._cls = TextRecognition
        self._mkldnn = enable_mkldnn
        self.model_name = model_name
        self._rec = self._make(model_name, enable_mkldnn)

    def _make(self, model_name: str, mkldnn: bool):
        try:
            return self._cls(model_name=model_name, enable_mkldnn=mkldnn)
        except Exception as e:
            if model_name != DEFAULT_REC_MODEL:
                print(f"[ml] rec model {model_name!r} unavailable ({type(e).__name__}: "
                      f"{e}); falling back to {DEFAULT_REC_MODEL}")
                self.model_name = DEFAULT_REC_MODEL
                return self._cls(model_name=DEFAULT_REC_MODEL, enable_mkldnn=mkldnn)
            raise

    def __call__(self, crops: list[np.ndarray], batch_size: int = 16) -> list[str]:
        texts: list[str] = []
        for start in range(0, len(crops), batch_size):
            batch = [cv2.cvtColor(np.ascontiguousarray(c), cv2.COLOR_RGB2BGR)
                     for c in crops[start:start + batch_size]]
            try:
                results = self._rec.predict(batch, batch_size=len(batch))
            except NotImplementedError as e:  # PIR+oneDNN executor bug -> retry
                if "Pir" not in str(e) and "onednn" not in str(e).lower():
                    raise
                self._rec = self._cls(model_name=self.model_name, enable_mkldnn=False)
                results = self._rec.predict(batch, batch_size=len(batch))
            for r in results:
                txt = r.get("rec_text") if hasattr(r, "get") else None
                texts.append((txt or "").strip())
        return texts


class MultilingualOcrEngine:
    """PP-OCRv5 detection -> reading order -> PP-OCRv5 recognition."""

    def __init__(self, lang: str = "ch", det_model: str = "PP-OCRv5_server_det",
                 rec_model: str | None = None, crop_pad: float = 0.0):
        self.lang = lang
        self.word_sep = "" if lang in _NO_SPACE_LANGS else " "
        # PP-OCR rec models are trained on tight crops (the detector already
        # unclips each box), so default to zero extra padding + cubic warp to
        # match PaddleOCR's own get_rotate_crop_image; padding double-expands the
        # box and drags neighboring glyphs in, hurting recognition on dense pages.
        self.crop_pad = crop_pad
        self.detector = Detector(det_model)
        self.recognizer = PaddleRecognizer(rec_model or rec_model_for(lang))

    def run(self, img_rgb: np.ndarray, batch_size: int = 16,
            make_visuals: bool = True) -> dict:
        """Full-page OCR. Returns text, per-line boxes/text, skew, timing, and
        (optionally) an overlay + side-by-side transcript composite."""
        t0 = time.perf_counter()
        if img_rgb.ndim == 2:
            img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_GRAY2RGB)
        elif img_rgb.shape[2] == 4:
            img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_RGBA2RGB)

        quads = self.detector(img_rgb)
        t_det = time.perf_counter()
        if len(quads) == 0:
            return {"text": "", "lines": [], "skew_deg": 0.0,
                    "seconds": {"detect": t_det - t0, "recognize": 0.0},
                    "note": "no text boxes detected"}

        lines, theta = pipeline.reading_order(quads)
        ordered = [pipeline.order_points(q) for q in quads]
        med_h = float(np.median([pipeline.quad_size(q)[1] for q in ordered]))

        # one crop per detected box; recognize all, then assemble in reading order
        crops: list[np.ndarray] = []
        owner_line: list[int] = []
        owner_pos: list[int] = []
        for li, line in enumerate(lines):
            for pos, i in enumerate(line.members):
                h = pipeline.quad_size(ordered[i])[1]
                c = pipeline.perspective_crop(img_rgb, ordered[i],
                                              pad_frac=self.crop_pad,
                                              allow_rot90=h > 2.2 * med_h,
                                              interp=cv2.INTER_CUBIC)
                if c is not None:
                    crops.append(c)
                    owner_line.append(li)
                    owner_pos.append(pos)

        box_texts = self.recognizer(crops, batch_size=batch_size)

        line_words: list[list[tuple[int, str]]] = [[] for _ in lines]
        for txt, li, pos in zip(box_texts, owner_line, owner_pos):
            if txt:
                line_words[li].append((pos, txt))
        line_texts = [self.word_sep.join(t for _, t in sorted(ws))
                      for ws in line_words]

        text = pipeline.assemble_text(lines, line_texts)
        t_rec = time.perf_counter()

        out = {
            "text": text,
            "lines": [{"text": line_texts[k],
                       "boxes": [quads[i].tolist() for i in lines[k].members]}
                      for k in range(len(lines))],
            "skew_deg": theta,
            "seconds": {"detect": t_det - t0, "recognize": t_rec - t_det},
            "n_boxes": int(len(quads)),
        }
        if make_visuals:
            out["overlay"] = pipeline.annotate(img_rgb, quads, lines)
            line_boxes = [[ordered[i] for i in lines[k].members]
                          for k in range(len(lines))]
            out["composite"] = pipeline.compose_transcript(
                img_rgb, line_boxes, line_texts)
        return out


In [ ]:
%%writefile ml_evaluate.py
"""Multilingual OCR evaluation: HI-RES (PP-OCRv5 det + reading-order + PP-OCRv5
rec) vs the stock PaddleOCR(lang=...) pipeline, scored on the SAME pages.

Dataset: XFUND (multilingual forms — ZH/JA/ES/FR/IT/DE/PT — CC-BY-4.0, from
microsoft/unilm), which ships per-segment text + boxes, so we get document-level
ground truth in reading order. Metrics (CER / WER / WordAcc / sec-per-img) are
imported from evaluate.py, so they are defined identically to the English
benchmark.

CJK note: Chinese/Japanese/Korean are scored space-free (whitespace removed from
both GT and prediction) — the standard character-level CER for scripts without
inter-word spaces. WER is not meaningful there and should be ignored; CER is the
headline. Latin languages keep spaces, so both CER and WER are meaningful.

Get the data (one folder per language):
    # download {lang}.{split}.json + the matching images from XFUND, unzip here
    xfund_zh/zh.val.json + xfund_zh/zh_val_0.jpg ...

    python multilingual/ml_evaluate.py --xfund xfund_zh --lang ch
    python multilingual/ml_evaluate.py --xfund xfund_fr --lang fr --gt-order geom
    python multilingual/ml_evaluate.py --xfund xfund_ja --lang ja --n 40 --no-baseline
"""
from __future__ import annotations

import argparse
import json
import re
import sys
from pathlib import Path

import cv2
import numpy as np

# import the root harness + the sibling engine
_HERE = Path(__file__).resolve().parent
_ROOT = _HERE.parent
for p in (str(_ROOT), str(_HERE)):
    if p not in sys.path:
        sys.path.insert(0, p)

from evaluate import (IMAGE_EXTS, NormCfg, Sample, evaluate_system,
                      format_table, save_csv)
from ml_engine import MultilingualOcrEngine, _NO_SPACE_LANGS, rec_model_for


# --------------------------------------------------------------------------
# XFUND loading
# --------------------------------------------------------------------------

def _xfund_docs(data) -> list[dict]:
    """XFUND json: usually {'documents': [...]}, sometimes a bare list."""
    if isinstance(data, dict):
        for key in ("documents", "data"):
            if isinstance(data.get(key), list):
                return data[key]
    if isinstance(data, list):
        return data
    return []


def _entity_box(ent: dict) -> list[float]:
    b = ent.get("box") or [0, 0, 0, 0]
    return [float(x) for x in b[:4]] if len(b) >= 4 else [0, 0, 0, 0]


def _doc_text(doc: dict, gt_order: str) -> str:
    """Page ground truth from a document's segments.

    gt_order='list' keeps the dataset's annotation order (the human reading
    order). gt_order='geom' re-sorts segments by (center-y, x) — a geometry
    reference, useful to check ordering on its own terms."""
    ents = [e for e in doc.get("document", []) if str(e.get("text", "")).strip()]
    if gt_order == "geom":
        ents.sort(key=lambda e: ((_entity_box(e)[1] + _entity_box(e)[3]) / 2.0,
                                 _entity_box(e)[0]))
    return "\n".join(str(e["text"]) for e in ents)


def _find_image(folder: Path, fname: str | None) -> Path | None:
    if not fname:
        return None
    cand = folder / fname
    if cand.is_file():
        return cand
    stem = Path(fname).stem
    for p in folder.rglob(Path(fname).name):
        if p.is_file():
            return p
    for ext in IMAGE_EXTS:
        for p in folder.rglob(stem + ext):
            if p.is_file():
                return p
    return None


def load_xfund(folder: str | Path, lang: str, split: str = "val",
               gt_order: str = "list", n: int | None = None) -> list[Sample]:
    folder = Path(folder)
    no_space = lang in _NO_SPACE_LANGS
    jsons = sorted(folder.rglob(f"*{split}*.json")) or sorted(folder.rglob("*.json"))
    samples: list[Sample] = []
    for jf in jsons:
        try:
            data = json.loads(jf.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            continue
        for doc in _xfund_docs(data):
            img = _find_image(folder, (doc.get("img") or {}).get("fname")
                              or doc.get("id"))
            if img is None:
                continue
            gt = _doc_text(doc, gt_order)
            if no_space:
                gt = re.sub(r"\s+", "", gt)
            if gt.strip():
                samples.append(Sample(name=Path(img).stem, image_path=img, gt=gt))
            if n is not None and len(samples) >= n:
                return samples
    return samples


# --------------------------------------------------------------------------
# predictors
# --------------------------------------------------------------------------

def hires_predict(engine: MultilingualOcrEngine, no_space: bool):
    def _f(img):
        text = engine.run(img, make_visuals=False)["text"]
        return re.sub(r"\s+", "", text) if no_space else text
    return _f


def _init_paddle(lang: str, det_model: str | None = None,
                 rec_model: str | None = None):
    """Stock PaddleOCR full pipeline (PIR+oneDNN guard applied).

    Pass det_model/rec_model to force the SAME models HI-RES uses — that turns the
    comparison into a controlled one where the only differences are HI-RES's
    reading order, keep-every-box, and crop method (a fair test of the layer).
    With both None, PaddleOCR loads its (heavier) defaults for `lang`."""
    from paddleocr import PaddleOCR
    base = dict(enable_mkldnn=False, use_doc_orientation_classify=False,
                use_doc_unwarping=False, use_textline_orientation=False)
    named = dict(base)
    if det_model:
        named["text_detection_model_name"] = det_model
    if rec_model:
        named["text_recognition_model_name"] = rec_model
    attempts = ([named] if (det_model or rec_model) else []) + [
        dict(base, lang=lang), base, dict(lang=lang), {}]
    last = None
    for kw in attempts:
        try:
            return PaddleOCR(**kw)
        except (TypeError, ValueError) as e:
            last = e
            continue
    raise RuntimeError(f"could not init PaddleOCR(lang={lang!r}): {last}")


def _paddle_texts(ocr, bgr) -> list[str]:
    if hasattr(ocr, "predict"):
        try:
            out: list[str] = []
            for r in ocr.predict(bgr):
                texts = r.get("rec_texts") if hasattr(r, "get") else None
                if texts:
                    out.extend(texts)
            return out
        except Exception:
            pass
    out = []
    for page in ocr.ocr(bgr) or []:
        for line in page or []:
            try:
                out.append(line[1][0])
            except (IndexError, TypeError):
                pass
    return out


def builtin_predict(lang: str, no_space: bool, det_model: str | None = None,
                    rec_model: str | None = None):
    ocr = _init_paddle(lang, det_model, rec_model)

    def _f(img):
        bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        texts = _paddle_texts(ocr, bgr)
        if no_space:
            return re.sub(r"\s+", "", "".join(texts))
        return "\n".join(texts)
    return _f


# --------------------------------------------------------------------------
# CLI
# --------------------------------------------------------------------------

def main() -> int:
    ap = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--xfund", required=True, help="folder with {lang}.{split}.json + images")
    ap.add_argument("--lang", required=True, help="PP-OCR language code (ch, ja, fr, de, ...)")
    ap.add_argument("--split", default="val", help="XFUND split substring (default: val)")
    ap.add_argument("--gt-order", choices=["list", "geom"], default="list",
                    help="ground-truth reading order: dataset list order (default) or geometric")
    ap.add_argument("--n", type=int, default=None, help="cap number of pages")
    ap.add_argument("--det-model", default="PP-OCRv5_mobile_det",
                    help="PP-OCR detection model for HI-RES (mobile = fast on CPU)")
    ap.add_argument("--rec-model", default=None, help="override PP-OCR rec model name")
    ap.add_argument("--controlled", action="store_true",
                    help="give stock PaddleOCR the SAME det+rec models as HI-RES, "
                         "isolating the reading-order layer (a fair, apples-to-apples test)")
    ap.add_argument("--no-baseline", action="store_true", help="skip stock PaddleOCR")
    ap.add_argument("--strip-punct", action="store_true")
    ap.add_argument("--csv", default="ml_eval_results.csv")
    args = ap.parse_args()

    no_space = args.lang in _NO_SPACE_LANGS
    samples = load_xfund(args.xfund, args.lang, split=args.split,
                         gt_order=args.gt_order, n=args.n)
    print(f"loaded {len(samples)} XFUND pages from {args.xfund} "
          f"[lang={args.lang}, split~{args.split}, gt-order={args.gt_order}]")
    if not samples:
        print("No (image, json) pairs found — check the folder and --split.")
        return 1

    norm = NormCfg(strip_punct=args.strip_punct)
    rec = args.rec_model or rec_model_for(args.lang)
    print(f"HI-RES models: det={args.det_model} rec={rec}"
          + ("  | stock: SAME models (controlled)" if args.controlled
             else "  | stock: PaddleOCR defaults"))
    engine = MultilingualOcrEngine(lang=args.lang, det_model=args.det_model,
                                   rec_model=args.rec_model)

    wm = not no_space  # CJK has no word boundaries -> WER/WordAcc meaningless
    scores = [evaluate_system(f"hires-ml[{args.lang}]", samples,
                              hires_predict(engine, no_space), norm=norm,
                              progress=True, word_metrics=wm)]
    if not args.no_baseline:
        det_m = args.det_model if args.controlled else None
        rec_m = rec if args.controlled else None
        try:
            scores.append(evaluate_system(
                f"paddle-stock[{args.lang}]", samples,
                builtin_predict(args.lang, no_space, det_m, rec_m),
                norm=norm, progress=True, word_metrics=wm))
        except Exception as e:
            print(f"  (stock PaddleOCR skipped: {type(e).__name__}: {e})")

    print("\n" + format_table(scores))
    if no_space:
        print("\n(CJK: no word boundaries — WER/WordAcc shown as '—'; CER is the metric.)")
    save_csv(scores, args.csv)
    print(f"per-page breakdown -> {args.csv}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 3. Get XFUND data + PP-OCRv6 models
[XFUND](https://github.com/doc-analysis/XFUND) (CC-BY-4.0) is multilingual forms with per-segment
text + boxes. **PP-OCRv6 medium is one unified model for 50 languages**, so a single det+rec pair
handles every language — no per-language rec model. `XFUND_LANG` only selects which dataset to
download. If the GitHub release URL 404s, see the HF fallback at the bottom.

In [ ]:
import urllib.request, zipfile
from pathlib import Path

XFUND_LANG = 'zh'      # zh | ja | es | fr | it | de | pt   (selects the dataset only)
PPOCR_LANG = {'zh': 'ch', 'ja': 'japan', 'es': 'es', 'fr': 'fr',
              'it': 'it', 'de': 'german', 'pt': 'pt'}[XFUND_LANG]

# PP-OCRv6 medium: unified 50-language det+rec (one pair for ALL languages).
# Lighter tiers if CPU is slow: 'PP-OCRv6_small_*' or 'PP-OCRv6_tiny_*'.
DET_MODEL = 'PP-OCRv6_medium_det'
REC_MODEL = 'PP-OCRv6_medium_rec'

DATA_DIR = f'xfund_{XFUND_LANG}'
Path(DATA_DIR).mkdir(exist_ok=True)
base = 'https://github.com/doc-analysis/XFUND/releases/download/v1.0'
for fn in (f'{XFUND_LANG}.val.json', f'{XFUND_LANG}.val.zip'):
    dst = Path(DATA_DIR, fn)
    if not dst.exists():
        print('downloading', fn, '...')
        urllib.request.urlretrieve(f'{base}/{fn}', dst)
with zipfile.ZipFile(Path(DATA_DIR, f'{XFUND_LANG}.val.zip')) as z:
    z.extractall(DATA_DIR)
print('XFUND', XFUND_LANG, 'ready in', DATA_DIR, '| det:', DET_MODEL, '| rec:', REC_MODEL)

## 4. Load data + visual sanity check
Run the engine on one page and look at the numbered boxes + reading-order transcript before
trusting metrics. `N_PAGES` is bigger here (40) for a meaningful sample; lower it if CPU is slow
(the `paddle device` print tells you — `cpu` is much slower than a T4).

In [ ]:
import matplotlib.pyplot as plt
import paddle
import ml_evaluate as M, ml_engine as ML

print('paddle device:', paddle.device.get_device(),
      ' (cpu = slow; use a T4 GPU runtime, or a lighter PP-OCRv6 tier)')

N_PAGES = 40          # bigger sample for a credible number; lower if CPU is slow
samples = M.load_xfund(DATA_DIR, PPOCR_LANG, split='val', n=N_PAGES)
print(len(samples), 'pages loaded')

engine = ML.MultilingualOcrEngine(lang=PPOCR_LANG, det_model=DET_MODEL, rec_model=REC_MODEL)
s = samples[0]
out = engine.run(s.image_rgb())

comp = out['composite']
ch, cw = comp.shape[:2]
plt.figure(figsize=(16, max(6, 16 * ch / cw)))
plt.imshow(comp); plt.axis('off')
plt.title(s.name + ' — page + reading-order transcript'); plt.show()
print('--- predicted (reading order) ---'); print(out['text'][:800])
print('\n--- ground truth ---'); print(s.gt[:800])

## 5. Clean forms: accuracy **and speed** (controlled, same PP-OCRv6 models)
Both run the **same** PP-OCRv6 det+rec, so the only differences are HI-RES's reading order,
keep-every-box, and **batched recognition**. Two numbers matter, both in the table (`sec/img`) and
summarized below it:
- **CER** — expect ~parity on clean upright forms (naive order is already correct).
- **Speed** — HI-RES collects every crop and runs the recognizer in batches, vs PaddleOCR's more
  sequential per-page pass; the printed **× faster** ratio + pages/s is the speed claim. Both
  predictors are **warmed up once** first so page-1 JIT isn't counted.

CJK is scored space-free → WER/WordAcc show as `—`; read CER.

In [ ]:
import evaluate as E

no_space = PPOCR_LANG in ML._NO_SPACE_LANGS
wm = not no_space          # CJK: no word boundaries -> skip WER/WordAcc
norm = E.NormCfg()

CONTROLLED = True          # stock uses the SAME PP-OCRv6 models as HI-RES (fair test)
stock_det, stock_rec = (DET_MODEL, REC_MODEL) if CONTROLLED else (None, None)

# build both predictors once (models load here) and REUSE them in §6
hires = M.hires_predict(engine, no_space)
try:
    stock = M.builtin_predict(PPOCR_LANG, no_space, stock_det, stock_rec)
except Exception as e:
    stock = None; print('stock init failed:', type(e).__name__, e)

# warm up once each so page-1 JIT/graph build is not counted in sec/img
_ = hires(samples[0].image_rgb())
if stock is not None:
    _ = stock(samples[0].image_rgb())

scores = [E.evaluate_system(f'hires-ml[{PPOCR_LANG}]', samples, hires,
                            norm=norm, progress=True, word_metrics=wm)]
if stock is not None:
    scores.append(E.evaluate_system(f'paddle-stock[{PPOCR_LANG}]', samples, stock,
                                    norm=norm, progress=True, word_metrics=wm))

print('\n' + E.format_table(scores))
if no_space:
    print('(CJK: no word boundaries -> WER/WordAcc are \u2014; CER is the metric.)')
if len(scores) == 2:
    h, st = scores
    ratio = st.sec_per_img / max(h.sec_per_img, 1e-9)
    faster = 'faster' if ratio >= 1 else 'SLOWER'
    print(f'\nSPEED  hires {h.sec_per_img:.2f}s/page  vs  stock {st.sec_per_img:.2f}s/page'
          f'  ->  {ratio:.1f}x {faster}'
          f'   ({1 / h.sec_per_img:.2f} vs {1 / st.sec_per_img:.2f} pages/s)')
E.save_csv(scores, 'ml_eval_results.csv')

## 6. Skew-robustness stress test — the reading-order payoff
Clean XFUND forms are upright, so naive top-to-bottom ordering is already right and §5 should be
~parity. The layer earns its keep when layout is **hard**. Here we **rotate each page** by a fixed
skew and re-score on the *same* ground truth (rotation doesn't change the words).

- **Stock PaddleOCR** orders detected boxes by raw position → under skew, lines from different
  rows interleave → wrong reading order → **CER rises**.
- **HI-RES** estimates the page skew and clusters/orders lines in *deskewed* coordinates → it
  should **stay flat**.

The gap between the two curves is precisely what a reading-order contribution to PaddleOCR would
close. If the curves *don't* diverge (stock already handles skew), there's no PR here — and that
is exactly the thing to find out before proposing one.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

def rotate_image(img, deg):
    if not deg:
        return img
    h, w = img.shape[:2]
    Mr = cv2.getRotationMatrix2D((w / 2, h / 2), deg, 1.0)
    return cv2.warpAffine(img, Mr, (w, h), borderValue=(255, 255, 255),
                          flags=cv2.INTER_LINEAR)

SKEW_ANGLES = [0, 5, 10]               # degrees; 0 = upright baseline
SKEW_N = min(15, len(samples))         # pages per angle (N x angles x 2 systems runs)
base = samples[:SKEW_N]

res = {'hires': [], 'stock': []}
for deg in SKEW_ANGLES:
    sset = [E.Sample(name=f'{s.name}_r{deg}', gt=s.gt,
                     image=rotate_image(s.image_rgb(), deg)) for s in base]
    h = E.evaluate_system(f'hires@{deg}', sset, hires, norm=norm, word_metrics=wm)
    res['hires'].append(h.cer)
    line = f'skew {deg:2d}deg: hires CER {h.cer:.1%}'
    if stock is not None:
        st = E.evaluate_system(f'stock@{deg}', sset, stock, norm=norm, word_metrics=wm)
        res['stock'].append(st.cer); line += f' | stock CER {st.cer:.1%}'
    print(line, flush=True)

plt.figure(figsize=(6.5, 4.5))
plt.plot(SKEW_ANGLES, res['hires'], 'o-', lw=2, label='HI-RES (deskews + reorders)')
if res['stock']:
    plt.plot(SKEW_ANGLES, res['stock'], 's--', lw=2, label='stock PaddleOCR')
plt.xlabel('page skew (degrees)'); plt.ylabel('CER  (lower is better)')
plt.title(f'Reading-order robustness to skew — XFUND {XFUND_LANG}, PP-OCRv6')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('skew_robustness.png', dpi=130); plt.show()
print('saved skew_robustness.png  ('
      'diverging curves = the reading-order layer adds value; flat-together = it does not)')

## 7. Clean-forms chart (CER per system)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
names = [s.system for s in scores]; x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(1.9 * len(names) + 3, 4))
ax.bar(x, [s.cer for s in scores], 0.5, color=['#2563eb', '#9aa0a6'][:len(names)])
for i, s in enumerate(scores):
    ax.text(i, s.cer, f'{s.cer:.1%}\n{s.sec_per_img:.1f}s', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('CER (lower better)'); ax.grid(axis='y', alpha=0.3)
ax.set_title(f'XFUND {XFUND_LANG} (upright): HI-RES vs stock, same PP-OCRv6 models')
plt.tight_layout(); plt.savefig('ml_comparison.png', dpi=130); plt.show()

## Notes — reading this honestly
- **§5 (clean forms)** measures recognition+crop parity and the batched-recognition speed edge.
  HI-RES ≈ stock on CER here is the *expected, good* outcome — it means the layer is lossless on
  easy layouts.
- **Speed claim — stay rigorous.** The `× faster` is *end-to-end* at equal models. HI-RES batches
  recognition (batch_size=16) across the whole page, while PaddleOCR's pipeline uses its own
  default rec batching — so part of the gap may be a *config* difference, not architecture. Before
  pitching speed as a PaddleOCR contribution, re-time stock with its rec batch raised; if the gap
  survives, it's a genuine throughput win (batched whole-page recognition).
- **§6 (skew)** is the decisive one. A widening CER gap (stock up, HI-RES flat) is reproducible
  evidence that PaddleOCR's box ordering breaks under skew and a geometry-based reorder fixes it —
  the seed of a focused PaddleOCR contribution (a reading-order post-process / pipeline option).
  No gap ⇒ no PR, and that is worth knowing.
- **Bigger / harder:** raise `N_PAGES`, add angles to `SKEW_ANGLES`, and try other `XFUND_LANG`
  values to show it generalizes across scripts.
- **XFUND download fallback**: if the GitHub release 404s, load via Hugging Face, e.g.
  `datasets.load_dataset('nielsr/XFUND', 'xfund.zh')`, write the images to `DATA_DIR`, and point
  `load_xfund` at it (it accepts any folder of `{lang}.val.json` + images).
- Paste both tables + the skew figure back, and we decide the PR form from the data.